In [33]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [34]:
GPDS_ROOT = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\GPDS150"
TRAIN_GENUINE = os.path.join(GPDS_ROOT, "train", "genuine")
TRAIN_FORGE   = os.path.join(GPDS_ROOT, "train", "forge")
TEST_ROOT     = os.path.join(GPDS_ROOT, "test")

print("Exists train/genuine:", os.path.exists(TRAIN_GENUINE))
print("Exists train/forge:", os.path.exists(TRAIN_FORGE))
print("Exists test:", os.path.exists(TEST_ROOT))

Exists train/genuine: True
Exists train/forge: True
Exists test: True


In [ ]:
def parse_gpds_filename(fname):
    """
    Returns: (writer_id:int, sample_id:int or None)
    Works for:
      c-001-03*.jpg
      cf-001-03*.jpg
    """
    m = re.search(r"(?:^|[^a-zA-Z0-9])c f?-?(\d+)-(\d+)", fname.replace(" ", ""), re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))

   
    m = re.search(r"c-0*(\d+)-0*(\d+)", fname, re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))

    m = re.search(r"cf-0*(\d+)-0*(\d+)", fname, re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))

   
    m = re.search(r"-0*(\d+)-0*(\d+)", fname)
    if m:
        return int(m.group(1)), int(m.group(2))

    return None, None

In [ ]:
def build_gpds150_df(gpds_root):
    rows = []
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

  
    train_genuine = os.path.join(gpds_root, "train", "genuine")
    train_forge   = os.path.join(gpds_root, "train", "forge")

    for label, folder in [("genuine", train_genuine), ("forgery", train_forge)]:
        for fname in os.listdir(folder):
            if Path(fname).suffix.lower() not in exts:
                continue
            wid, sid = parse_gpds_filename(fname)
            if wid is None:
                continue
            rows.append({
                "dataset": "GPDS150",
                "split": "train",
                "writer_id": wid,
                "sample_id": sid,
                "label": label,
                "path": os.path.join(folder, fname),
                "filename": fname
            })

  
    test_root = os.path.join(gpds_root, "test")
    for writer in sorted(os.listdir(test_root)):
        writer_dir = os.path.join(test_root, writer)
        if not os.path.isdir(writer_dir):
            continue

        for subfolder, label in [("genuine", "genuine"), ("forge", "forgery")]:
            subdir = os.path.join(writer_dir, subfolder)
            if not os.path.exists(subdir):
                continue

            for fname in os.listdir(subdir):
                if Path(fname).suffix.lower() not in exts:
                    continue
                wid, sid = parse_gpds_filename(fname)
                if wid is None:
                    
                    wid = int(writer)
                rows.append({
                    "dataset": "GPDS150",
                    "split": "test",
                    "writer_id": int(wid),
                    "sample_id": sid,
                    "label": label,
                    "path": os.path.join(subdir, fname),
                    "filename": fname
                })

    return pd.DataFrame(rows)

valid_gpds = build_gpds150_df(GPDS_ROOT)

print("Total images:", len(valid_gpds))
print(valid_gpds.groupby(["split","label"]).size())
print("Unique writers (all):", valid_gpds["writer_id"].nunique())
display(valid_gpds.head())

Total images: 8100
split  label  
test   forgery    2100
       genuine    1200
train  forgery    2400
       genuine    2400
dtype: int64
Unique writers (all): 150


,dataset,split,writer_id,sample_id,label,path,filename
0,GPDS150,train,1,1,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-01 (Copy).jpg
1,GPDS150,train,1,2,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-02 (Copy).jpg
2,GPDS150,train,1,3,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-03 (Copy).jpg
3,GPDS150,train,1,4,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-04 (Copy).jpg
4,GPDS150,train,1,5,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-05 (Copy).jpg


In [ ]:
train_df = valid_gpds[valid_gpds["split"] == "train"].copy()
test_df  = valid_gpds[valid_gpds["split"] == "test"].copy()

print("Train writers:", train_df["writer_id"].nunique())
print("Test writers:", test_df["writer_id"].nunique())


train_counts = train_df.groupby(["writer_id","label"]).size().unstack(fill_value=0)
display(train_counts.head())

print("Writers with >=2 genuine:", int((train_counts["genuine"] >= 2).sum()))
print("Writers with >=1 forgery:", int((train_counts["forgery"] >= 1).sum()))

Train writers: 150
Test writers: 150


label,forgery,genuine
writer_id,,
1,16,16
2,16,16
3,16,16
4,16,16
5,16,16


Writers with >=2 genuine: 150
Writers with >=1 forgery: 150


In [38]:
all_train_writers = np.array(sorted(train_df["writer_id"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(all_train_writers)

val_ratio = 0.2
n_val = int(round(len(all_train_writers) * val_ratio))

val_writers = set(all_train_writers[:n_val])
train_writers = set(all_train_writers[n_val:])
test_writers = set(test_df["writer_id"].unique())

print("Train writers:", len(train_writers))
print("Val writers:", len(val_writers))
print("Test writers:", len(test_writers))
print("Overlap train-val:", len(train_writers & val_writers))

Train writers: 120
Val writers: 30
Test writers: 150
Overlap train-val: 0


In [39]:
def build_pools(df_subset):
    genuine_by_writer = {}
    forgery_by_writer = {}

    for wid, g in df_subset.groupby("writer_id"):
        gg = g[g["label"] == "genuine"]["path"].tolist()
        ff = g[g["label"] == "forgery"]["path"].tolist()

        if len(gg) >= 2:
            genuine_by_writer[wid] = gg
        if len(ff) >= 1:
            forgery_by_writer[wid] = ff

    return genuine_by_writer, forgery_by_writer

def generate_pairs(df, writer_set, n_pairs=80000, seed=1, neg_mix=0.9):
    df_subset = df[df["writer_id"].isin(writer_set)].copy()
    genuine_by_writer, forgery_by_writer = build_pools(df_subset)

    writers = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))

    if len(writers) < 2:
        raise ValueError(f"Need >=2 writers with >=2 genuine each. Found {len(writers)}.")
    if len(writers_with_forg) == 0:
        raise ValueError("Need at least 1 writer with forgeries.")

    rng = np.random.default_rng(seed)
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same  = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    pairs = []

    # positives: genuine-genuine same writer
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        pairs.append({"path_a": g[i], "path_b": g[j], "label": 1})

    # negatives: genuine-forgery same writer
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        i = rng.integers(0, len(g))
        j = rng.integers(0, len(f))
        pairs.append({"path_a": g[i], "path_b": f[j], "label": 0})

    # negatives: genuine-genuine cross writer
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1 = genuine_by_writer[w1]
        g2 = genuine_by_writer[w2]
        i = rng.integers(0, len(g1))
        j = rng.integers(0, len(g2))
        pairs.append({"path_a": g1[i], "path_b": g2[j], "label": 0})

    return pd.DataFrame(pairs).sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_pairs = generate_pairs(train_df, train_writers, n_pairs=80000, seed=1, neg_mix=0.9)
val_pairs   = generate_pairs(train_df, val_writers,   n_pairs=20000, seed=2, neg_mix=0.9)
test_pairs  = generate_pairs(test_df,  test_writers,  n_pairs=20000, seed=3, neg_mix=0.9)

print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs), "Test pairs:", len(test_pairs))
print("Train balance:\n", train_pairs["label"].value_counts(normalize=True))

Train pairs: 80000 Val pairs: 20000 Test pairs: 20000
Train balance:
 label
1    0.5
0    0.5
Name: proportion, dtype: float64


In [ ]:
IMG_SIZE = (224, 224)

def load_preprocess(path):
    path = path.numpy().decode("utf-8")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Failed to read: {path}")

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)

    return img

def tf_load_preprocess(path):
    img = tf.py_function(load_preprocess, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])
    return img

def tf_load_preprocess_resnet(path):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=1)

    img = tf.image.resize(img, IMG_SIZE)

    
    img = tf.image.grayscale_to_rgb(img)

    img = tf.cast(img, tf.float32)

   
    img = tf.keras.applications.resnet.preprocess_input(img)

    return img

def make_pair_dataset_resnet(pairs_df, batch_size=32, shuffle=True):
    a = pairs_df["path_a"].astype(str).values
    b = pairs_df["path_b"].astype(str).values
    y = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))

    def map_fn(pa, pb, lab):
        ia = tf_load_preprocess_resnet(pa)
        ib = tf_load_preprocess_resnet(pb)
        return (ia, ib), lab

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(4000, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_pair_dataset_resnet(train_pairs, batch_size=32, shuffle=True)
val_ds   = make_pair_dataset_resnet(val_pairs,   batch_size=32, shuffle=False)
test_ds  = make_pair_dataset_resnet(test_pairs,  batch_size=32, shuffle=False)

(xb, yb) = next(iter(train_ds))
print(xb[0].shape, xb[1].shape, yb.shape)


(32, 224, 224, 3) (32, 224, 224, 3) (32,)


In [41]:
class SquaredEuclidean(layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True)

In [ ]:
class ContrastiveLoss(keras.losses.Loss):
    def __init__(self, margin=1.0, name="contrastive_loss"):
        super().__init__(name=name)
        self.margin = margin

    def call(self, y_true, d_sq):
    
        y_true = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
        d_sq = tf.cast(d_sq, tf.float32)

        pos = y_true * d_sq
        neg = (1.0 - y_true) * tf.square(tf.maximum(0.0, self.margin - tf.sqrt(d_sq + 1e-9)))
        return tf.reduce_mean(pos + neg)

In [43]:
@tf.function
def mean_pos_dist(y_true, d_sq):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    d = tf.sqrt(tf.cast(tf.reshape(d_sq, [-1]), tf.float32) + 1e-9)
    pos = tf.boolean_mask(d, tf.equal(y_true, 1.0))
    return tf.cond(tf.size(pos) > 0, lambda: tf.reduce_mean(pos), lambda: tf.constant(0.0))

@tf.function
def mean_neg_dist(y_true, d_sq):
    y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    d = tf.sqrt(tf.cast(tf.reshape(d_sq, [-1]), tf.float32) + 1e-9)
    neg = tf.boolean_mask(d, tf.equal(y_true, 0.0))
    return tf.cond(tf.size(neg) > 0, lambda: tf.reduce_mean(neg), lambda: tf.constant(0.0))

In [44]:
from tensorflow.keras.applications import ResNet50

EMB_DIM = 128

def build_resnet_embedding(input_shape=(224, 224, 3), emb_dim=128, train_backbone=False):
    inp = keras.Input(shape=input_shape)

    backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_tensor=inp
    )
    backbone.trainable = train_backbone

    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(emb_dim)(x)

    # L2 normalize WITHOUT Lambda issues
    x = layers.UnitNormalization(axis=1, name="l2_norm")(x)

    return keras.Model(inp, x, name="resnet_embedding")

embedding_net = build_resnet_embedding(input_shape=(224,224,3), emb_dim=EMB_DIM, train_backbone=False)
embedding_net.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 85s 1us/step


Model: "resnet_embedding"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,145,152 (92.11 MB)

 Trainable params: 557,440 (2.13 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
def build_siamese(embedding_net, input_shape=(224,224,3)):
    a = keras.Input(shape=input_shape, name="img_a")
    b = keras.Input(shape=input_shape, name="img_b")

    ea = embedding_net(a)
    eb = embedding_net(b)

    d_sq = SquaredEuclidean(name="d_sq")([ea, eb])  
    return keras.Model([a, b], d_sq, name="siamese_resnet")

siamese_model = build_siamese(embedding_net, input_shape=(224,224,3))
siamese_model.summary()

Model: "siamese_resnet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img_a (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img_b (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet_embedding    │ (None, 128)       │ 24,145,152 │ img_a[0][0],      │
│ (Functional)        │                   │            │ img_b[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ d_sq                │ (None, 1)         │          0 │ resnet_embedding… │
│ (SquaredEuclidean)  │                   │            │ resnet_embedding… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,145,152 (92.11 MB)

 Trainable params: 557,440 (2.13 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [46]:
loss_fn = ContrastiveLoss(margin=1.0)

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=loss_fn,
    metrics=[mean_pos_dist, mean_neg_dist]
)

ckpt_path = r"W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\GPDS150\resnet_baseline_best.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-6),
]

history_1 = siamese_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

Epoch 1/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 3899s 2s/step - loss: 0.1192 - mean_neg_dist: 0.9237 - mean_pos_dist: 0.3143 - val_loss: 0.1677 - val_mean_neg_dist: 1.0065 - val_mean_pos_dist: 0.3613 - learning_rate: 3.0000e-04
Epoch 2/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 3806s 2s/step - loss: 0.0635 - mean_neg_dist: 1.2270 - mean_pos_dist: 0.1993 - val_loss: 0.2023 - val_mean_neg_dist: 1.0519 - val_mean_pos_dist: 0.3776 - learning_rate: 3.0000e-04
Epoch 3/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 3806s 2s/step - loss: 0.0399 - mean_neg_dist: 1.4441 - mean_pos_dist: 0.1449 - val_loss: 0.2906 - val_mean_neg_dist: 1.2806 - val_mean_pos_dist: 0.4994 - learning_rate: 1.5000e-04


In [ ]:

backbone = embedding_net.get_layer("resnet50")
backbone.trainable = True


for layer in backbone.layers[:-30]:
    layer.trainable = False

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss_fn,
    metrics=[mean_pos_dist, mean_neg_dist]
)

history_2 = siamese_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

ValueError: No such layer: resnet50. Existing layers are: ['input_layer', 'conv1_pad', 'conv1_conv', 'conv1_bn', 'conv1_relu', 'pool1_pad', 'pool1_pool', 'conv2_block1_1_conv', 'conv2_block1_1_bn', 'conv2_block1_1_relu', 'conv2_block1_2_conv', 'conv2_block1_2_bn', 'conv2_block1_2_relu', 'conv2_block1_0_conv', 'conv2_block1_3_conv', 'conv2_block1_0_bn', 'conv2_block1_3_bn', 'conv2_block1_add', 'conv2_block1_out', 'conv2_block2_1_conv', 'conv2_block2_1_bn', 'conv2_block2_1_relu', 'conv2_block2_2_conv', 'conv2_block2_2_bn', 'conv2_block2_2_relu', 'conv2_block2_3_conv', 'conv2_block2_3_bn', 'conv2_block2_add', 'conv2_block2_out', 'conv2_block3_1_conv', 'conv2_block3_1_bn', 'conv2_block3_1_relu', 'conv2_block3_2_conv', 'conv2_block3_2_bn', 'conv2_block3_2_relu', 'conv2_block3_3_conv', 'conv2_block3_3_bn', 'conv2_block3_add', 'conv2_block3_out', 'conv3_block1_1_conv', 'conv3_block1_1_bn', 'conv3_block1_1_relu', 'conv3_block1_2_conv', 'conv3_block1_2_bn', 'conv3_block1_2_relu', 'conv3_block1_

In [ ]:

embedding_net.trainable = True

for layer in embedding_net.layers:
   
    layer.trainable = False

    
    if (
        layer.name.startswith("conv5_")
        or layer.name in ["global_average_pooling2d", "dense", "dropout", "dense_1", "l2_norm"]
    ):
        layer.trainable = True


print("Trainable layers:", sum([int(l.trainable) for l in embedding_net.layers]), "/", len(embedding_net.layers))

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss_fn,
    metrics=[mean_pos_dist, mean_neg_dist]
)

history_2 = siamese_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

Trainable layers: 37 / 180
Epoch 1/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 5055s 2s/step - loss: 0.0509 - mean_neg_dist: 1.3227 - mean_pos_dist: 0.1643 - val_loss: 0.2621 - val_mean_neg_dist: 1.1560 - val_mean_pos_dist: 0.4812 - learning_rate: 1.0000e-04
Epoch 2/5
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 35816s 14s/step - loss: 0.0160 - mean_neg_dist: 1.5185 - mean_pos_dist: 0.0907 - val_loss: 0.3520 - val_mean_neg_dist: 1.1812 - val_mean_pos_dist: 0.6249 - learning_rate: 5.0000e-05


In [ ]:
def get_distances(model, ds):
    d_all, y_all = [], []
    for (x, y) in ds:
       
        d_sq = model.predict(x, verbose=0).reshape(-1)
        d = np.sqrt(np.maximum(d_sq, 0.0))  
        d_all.append(d)
        y_all.append(y.numpy().reshape(-1))
    return np.concatenate(d_all), np.concatenate(y_all)

In [ ]:
def compute_far_frr(d, y, thresholds):
    y = y.astype(int)
    neg = (y == 0)  
    pos = (y == 1)  
    far, frr = [], []
    for t in thresholds:
        far.append(np.mean(d[neg] <= t))  
        frr.append(np.mean(d[pos] >  t))  
    return np.array(far), np.array(frr)

val_d, val_y = get_distances(siamese_model, val_ds)

thresholds = np.linspace(val_d.min(), val_d.max(), 800)
far, frr = compute_far_frr(val_d, val_y, thresholds)

eer_idx = np.argmin(np.abs(far - frr))
eer = (far[eer_idx] + frr[eer_idx]) / 2
best_t = thresholds[eer_idx]

print("Val dist min/mean/max:", val_d.min(), val_d.mean(), val_d.max())
print("Validation EER:", float(eer))
print("Best threshold:", float(best_t))
print("FAR@EER:", float(far[eer_idx]), "FRR@EER:", float(frr[eer_idx]))

Val dist min/mean/max: 0.01214459 0.81863576 1.8792293
Validation EER: 0.2541
Best threshold: 0.7038304805755615
FAR@EER: 0.2542 FRR@EER: 0.254


In [52]:
test_d, test_y = get_distances(siamese_model, test_ds)
test_y = test_y.astype(int)

neg = (test_y == 0)
pos = (test_y == 1)

test_far = np.mean(test_d[neg] <= best_t)
test_frr = np.mean(test_d[pos] >  best_t)
test_avg = (test_far + test_frr) / 2

print("Test dist min/mean/max:", test_d.min(), test_d.mean(), test_d.max())
print("Test FAR (val threshold):", float(test_far))
print("Test FRR (val threshold):", float(test_frr))
print("Test avg error:", float(test_avg))

Test dist min/mean/max: 0.0066554034 0.73868865 1.8829235
Test FAR (val threshold): 0.2193
Test FRR (val threshold): 0.0873
Test avg error: 0.1533


In [48]:
save_dir = r"W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\GPDS150\resnet"
os.makedirs(save_dir, exist_ok=True)

siamese_model.save(os.path.join(save_dir, "gpds_resnet_siamese.keras"))
embedding_net.save(os.path.join(save_dir, "gpds_resnet_embedding.keras"))